In [ ]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import cupy as cp
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error
)
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression

# 修改：正确导入cuML的相关模块
from cuml.ensemble import RandomForestClassifier as cuRF, RandomForestRegressor as cuRFR
from cuml.linear_model import LogisticRegression as cuLogisticRegression, LinearRegression as cuLinearRegression
from cuml.svm import SVC as cuSVC, SVR as cuSVR
from cuml.model_selection import train_test_split as cu_train_test_split  # 修正导入
from cuml.metrics import accuracy_score as cu_accuracy_score
from cuml import __version__ as cuml_version
import warnings
warnings.filterwarnings("ignore", message="Random state is currently ignored by probabilistic SVC")

# 新增：梯度提升模型导入
try:
    from cuml.ensemble import GradientBoostingClassifier as cuGBC, GradientBoostingRegressor as cuGBR
    HAS_GRADIENT_BOOSTING = True
except ImportError:
    print("cuML Gradient Boosting 不可用，将跳过该模型")
    HAS_GRADIENT_BOOSTING = False

# XGBoost导入
import xgboost as xgb

# 其他导入
from datetime import datetime
from pandas.api.types import CategoricalDtype
import traceback
from sklearn.base import clone, BaseEstimator
from sklearn.model_selection import KFold
from typing import Dict, Any, Tuple, Optional, List
import warnings
import json
warnings.filterwarnings('ignore')

# 设置环境变量
os.environ['XGBOOST_DEVICE_MEMORY_LIMIT'] = '10GB'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# 新增：导入绘图相关模块
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm

# 新增：导入Word文档保存模块
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT

# 新增：用于深度复制PyTorch模型
import copy

# ========================
# GPU资源清理增强模块
# ========================
def clear_gpu_resources():
    """全面清理GPU资源并添加延迟"""
    gc.collect()
    time.sleep(0.5)
    
    if 'cp' in globals():
        try:
            cp._default_memory_pool.free_all_blocks()
            cp.get_default_memory_pool().free_all_blocks()
        except Exception: 
            pass
        
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass

def monitor_gpu_memory(threshold=0.8):
    """监控GPU内存使用，超过阈值则清理"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated()
        total = torch.cuda.get_device_properties(0).total_memory
        ratio = allocated / total
        if ratio > threshold:
            print(f"GPU内存使用率超过{threshold*100:.1f}% ({allocated/1e9:.2f}GB/{total/1e9:.2f}GB)，执行清理...")
            clear_gpu_resources()
            return True
    return False

# ========================
# 辅助函数：PyTorch版本的自助法标准误
# ========================
def bootstrap_se_torch(X: np.ndarray, y: np.ndarray, n_bootstrap: int = 1000) -> Tuple[float, float, float]:
    """
    使用自助法计算标准误和置信区间（PyTorch优化版）
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_samples = len(y)
    
    bootstrap_effects = []
    
    for i in range(n_bootstrap):
        # 生成自助样本
        indices = np.random.choice(n_samples, n_samples, replace=True)
        X_boot = torch.tensor(X[indices], dtype=torch.float32, device=device)
        y_boot = torch.tensor(y[indices], dtype=torch.float32, device=device)
        
        # 使用正规方程求解
        with torch.no_grad():
            XtX = X_boot.t() @ X_boot
            Xty = X_boot.t() @ y_boot.reshape(-1, 1)
            
            # 添加小常数防止奇异矩阵
            XtX += torch.eye(X_boot.shape[1], device=device) * 1e-6
            
            coefficients = torch.linalg.solve(XtX, Xty)
            effect = coefficients[1].item() if coefficients.shape[0] > 1 else coefficients[0].item()
        
        bootstrap_effects.append(effect)
    
    # 计算标准误和置信区间
    bootstrap_effects = np.array(bootstrap_effects)
    se = np.std(bootstrap_effects, ddof=1)
    ci_lower = np.percentile(bootstrap_effects, 2.5)
    ci_upper = np.percentile(bootstrap_effects, 97.5)
    
    return se, ci_lower, ci_upper

# ========================
# 辅助函数：PyTorch版本的聚类稳健标准误
# ========================
def calculate_cluster_robust_se_torch(X: torch.Tensor, 
                                     y: torch.Tensor, 
                                     residuals: torch.Tensor, 
                                     cluster_ids: np.ndarray) -> float:
    """
    计算聚类稳健标准误（PyTorch版本）
    """
    device = X.device
    unique_clusters = np.unique(cluster_ids)
    n_clusters = len(unique_clusters)
    
    scores = []
    
    for cluster in unique_clusters:
        cluster_mask = (cluster_ids == cluster)
        X_cluster = X[cluster_mask]
        residuals_cluster = residuals[cluster_mask]
        
        # 计算聚类得分
        cluster_score = X_cluster.t() @ residuals_cluster
        scores.append(cluster_score.cpu().numpy())
    
    # 计算聚类稳健方差估计
    scores_matrix = np.array(scores)
    mean_score = np.mean(scores_matrix, axis=0)
    cluster_variance = (scores_matrix - mean_score).T @ (scores_matrix - mean_score) / (n_clusters - 1)
    
    # 计算标准误
    XtX_inv = torch.linalg.inv(X.t() @ X).cpu().numpy()
    cluster_vcov = XtX_inv @ cluster_variance @ XtX_inv
    se = np.sqrt(np.diag(cluster_vcov))[1]
    
    return se

# ========================
# 新增：第一阶段模型工厂类
# ========================
class FirstStageModelFactory:
    """第一阶段模型工厂类，用于创建和配置不同的机器学习模型"""
    def __init__(self):
        self.model_configs = {
            'RandomForest': {
                'classification': lambda: cuRF(n_estimators=200, max_depth=20, random_state=42),
                'regression': lambda: cuRFR(n_estimators=200, max_depth=20, random_state=42)
            },
            'LogisticRegression': {
                'classification': lambda: cuLogisticRegression(penalty='l2', C=1.0, max_iter=1000),
                'regression': lambda: cuLinearRegression(fit_intercept=True, algorithm='eig')
            },
            'GradientBoosting': {
                'classification': lambda: cuGBC(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None,
                'regression': lambda: cuGBR(n_estimators=200, max_depth=6, random_state=42) if HAS_GRADIENT_BOOSTING else None
            },
            'SVC': {
                'classification': lambda: cuSVC(kernel='rbf', C=1.0, probability=True, random_state=42),
                'regression': lambda: cuSVR(kernel='rbf', C=1.0)
            },
            'XGBoost': {
                'classification': lambda: xgb.XGBClassifier(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist'),
                'regression': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=6, random_state=42, tree_method='gpu_hist')
            },
            'MLP': {
                'classification': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='classification'),
                'regression': lambda input_size: PyTorchMLP(input_size=input_size, output_size=1, task_type='regression')
            }
        }
    
    def get_model(self, model_name, task_type, input_size=None):
        """
        根据模型名称和任务类型获取模型实例
        """
        if model_name not in self.model_configs:
            raise ValueError(f"不支持的模型类型: {model_name}")
        
        if task_type not in self.model_configs[model_name]:
            raise ValueError(f"模型 {model_name} 不支持任务类型: {task_type}")
        
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            if input_size is None:
                raise ValueError("MLP模型需要指定input_size参数")
            return self.model_configs[model_name][task_type](input_size)
        else:
            model = self.model_configs[model_name][task_type]()
            if model is None:
                raise ValueError(f"模型 {model_name} 不可用，可能缺少依赖")
            return model

# ========================
# 新增：PyTorch MLP模型（GPU版本）
# ========================
class PyTorchMLP(nn.Module):
    """PyTorch多层感知机模型（GPU加速版），用于替代cuML的MLP"""

    def __init__(self, input_size, hidden_sizes=[100, 50], output_size=1, 
                 task_type='classification', dropout_rate=0.1):
        super(PyTorchMLP, self).__init__()
        self.task_type = task_type
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 构建网络层
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        
        # 根据任务类型添加输出层激活函数
        if task_type == 'classification':
            layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers).to(self.device)
        
    def forward(self, x):
        return self.network(x)
    
    def fit(self, X, y, epochs=100, batch_size=256, lr=0.001):
        """训练模型"""
        self.train()
        
        # 转换数据为PyTorch张量
        if not isinstance(X, torch.Tensor):
            X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
        else:
            X_tensor = X.to(self.device)
            
        if not isinstance(y, torch.Tensor):
            y_tensor = torch.tensor(y, dtype=torch.float32, device=self.device)
        else:
            y_tensor = y.to(self.device)
        
        # 确保输出维度匹配
        if self.task_type == 'classification':
            y_tensor = y_tensor.view(-1, 1)
        
        # 定义损失函数和优化器
        if self.task_type == 'classification':
            criterion = nn.BCELoss()
        else:
            criterion = nn.MSELoss()
        
        optimizer = optim.Adam(self.parameters(), lr=lr)
        
        # 训练循环
        dataset = torch.utils.data.TensorDataset(X_tensor, y_tensor)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_X, batch_y in dataloader:
                optimizer.zero_grad()
                outputs = self(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            if (epoch + 1) % 20 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")
    
    def predict(self, X):
        """预测"""
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            predictions = self(X_tensor)
            
            if self.task_type == 'classification':
                return (predictions > 0.5).float().cpu().numpy().flatten()
            else:
                return predictions.cpu().numpy().flatten()
    
    def predict_proba(self, X):
        """预测概率（仅分类任务）"""
        if self.task_type != 'classification':
            raise ValueError("predict_proba仅适用于分类任务")
        
        self.eval()
        with torch.no_grad():
            if not isinstance(X, torch.Tensor):
                X_tensor = torch.tensor(X, dtype=torch.float32, device=self.device)
            else:
                X_tensor = X.to(self.device)
                
            return self(X_tensor).cpu().numpy()

# ========================
# 使用PyTorch进行DML分析（优化版）
# ========================
from scipy import stats
def second_stage_dml_analysis(t_residuals, y_residuals, X_second=None, robust_method='bootstrap', n_bootstrap=1000):
    """
    执行DML的第二阶段分析，计算处理效应及统计检验指标。
    
    参数:
        t_residuals (torch.Tensor或np.array): 第一阶段处理模型的残差。
        y_residuals (torch.Tensor或np.array): 第一阶段结果模型的残差。
        X_second (torch.Tensor或np.array, optional): 用于异质性分析的协变量。默认为None。
        robust_method (str): 稳健标准误方法，'bootstrap'或'hc1'。默认为'bootstrap'。
        n_bootstrap (int): Bootstrap重复次数，仅当robust_method='bootstrap'时有效。默认为1000。
    
    返回:
        dict: 包含效应值、标准误、置信区间、t统计量、p值、F统计量、F检验p值等的字典。
    """
    # 确保将Tensor转换为NumPy数组
    if torch.is_tensor(t_residuals):
        t_residuals = t_residuals.cpu().numpy() if t_residuals.is_cuda else t_residuals.numpy()
    if torch.is_tensor(y_residuals):
        y_residuals = y_residuals.cpu().numpy() if y_residuals.is_cuda else y_residuals.numpy()
    if X_second is not None and torch.is_tensor(X_second):
        X_second = X_second.cpu().numpy() if X_second.is_cuda else X_second.numpy()
    
    n = len(y_residuals)
    
    # 确保残差是NumPy数组并调整为正确的形状
    t_residuals = np.asarray(t_residuals).reshape(-1, 1)
    y_residuals = np.asarray(y_residuals).reshape(-1, 1)
    
    # 第二阶段回归：Y_residuals ~ theta * T_residuals + [其他交互项]
    if X_second is not None:
        # 如果有协变量用于异质性分析（CATE），包含交互项
        X_second = np.asarray(X_second)
        # 构建设计矩阵: [T_residuals, T_residuals * X_second]
        interaction_terms = t_residuals * X_second
        design_matrix = np.column_stack((t_residuals, interaction_terms))
    else:
        # 只估计ATE
        design_matrix = t_residuals
    
    # 使用OLS估计系数
    coefficients, _, _, _ = np.linalg.lstsq(design_matrix, y_residuals, rcond=None)
    
    # 提取处理效应（ATE是第一个系数）
    effect = coefficients[0][0] if len(coefficients) > 0 else 0.0
    
    # 计算预测值和残差
    y_pred = np.dot(design_matrix, coefficients)
    residuals = y_residuals - y_pred
    
    # 初始化变量
    se = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    
    # 计算标准误和置信区间
    if robust_method == 'bootstrap':
        bootstrap_effects = []
        for _ in range(n_bootstrap):
            indices = np.random.choice(n, n, replace=True)
            t_boot = t_residuals[indices]
            y_boot = y_residuals[indices]
            if X_second is not None:
                X_second_boot = X_second[indices]
                interaction_boot = t_boot * X_second_boot
                design_boot = np.column_stack((t_boot, interaction_boot))
            else:
                design_boot = t_boot
            coef_boot, _, _, _ = np.linalg.lstsq(design_boot, y_boot, rcond=None)
            bootstrap_effects.append(coef_boot[0][0] if len(coef_boot) > 0 else 0.0)
        se = np.std(bootstrap_effects, ddof=1) if bootstrap_effects else 0.0
    else:
        # 使用HC1稳健标准误的简化实现
        mse = np.sum(residuals**2) / (n - design_matrix.shape[1])
        xtx_inv = np.linalg.inv(np.dot(design_matrix.T, design_matrix))
        se_matrix = mse * xtx_inv
        se = np.sqrt(np.diag(se_matrix)[0]) if se_matrix.size > 0 else 0.0
    
    # 计算置信区间
    ci_lower = effect - 1.96 * se if se > 0 else effect
    ci_upper = effect + 1.96 * se if se > 0 else effect
    
    # 计算t统计量和p值 (只有在se>0时才有意义)
    if se > 0:
        t_statistic = effect / se
        df = n - design_matrix.shape[1]  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    
    # 计算F统计量和p值 (检验整个第二阶段模型是否显著)
    ss_total = np.sum((y_residuals - np.mean(y_residuals))**2)
    ss_residual = np.sum(residuals**2)
    if ss_residual > 0 and design_matrix.shape[1] > 0:
        ss_explained = ss_total - ss_residual
        df_model = design_matrix.shape[1]
        df_resid = n - df_model
        f_statistic = (ss_explained / df_model) / (ss_residual / df_resid) if df_resid > 0 else 0.0
        f_p_value = 1 - stats.f.cdf(f_statistic, df_model, df_resid) if df_resid > 0 else 1.0
    
    # 返回所有结果
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': n - design_matrix.shape[1] if design_matrix.shape[1] > 0 else n
    }

def torch_dml_with_pretrained(X: np.ndarray, 
                              y: np.ndarray, 
                              t: np.ndarray, 
                              model_y: BaseEstimator, 
                              model_t: BaseEstimator, 
                              use_robust_se: bool = True, 
                              robust_method: str = 'bootstrap',
                              n_bootstrap: int = 1000,
                              cluster_ids: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """
    使用预训练的第一阶段模型执行DML的第二阶段分析（残差回归）
    
    参数:
        X: 协变量矩阵
        y: 结果变量
        t: 处理变量
        model_y: 预训练的结果变量模型
        model_t: 预训练的处理变量模型
        use_robust_se: 是否使用稳健标准误
        robust_method: 稳健标准误计算方法 ('bootstrap' 或 'hc1')
        n_bootstrap: 自助法重采样次数
        cluster_ids: 聚类ID（用于聚类稳健标准误）
    
    返回:
        包含效应值、标准误、置信区间等结果的字典
    """
    print("执行DML第二阶段分析（残差回归）...")
    
    # 初始化所有变量
    effect = 0.0
    se = 0.0
    ci_lower = 0.0
    ci_upper = 0.0
    t_statistic = 0.0
    p_value = 1.0
    f_statistic = 0.0
    f_p_value = 1.0
    df = 0
    
    # 转换为PyTorch张量并移至GPU（如果可用）
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 确保输入为NumPy数组
    X_np = X if isinstance(X, np.ndarray) else cp.asnumpy(X)
    y_np = y if isinstance(y, np.ndarray) else cp.asnumpy(y)
    t_np = t if isinstance(t, np.ndarray) else cp.asnumpy(t)
    
    # 第一阶段预测：获取Y和T的预测值
    print("计算第一阶段预测值...")
    with torch.no_grad():
        # 使用第一阶段模型进行预测
        y_pred = model_y.predict(X_np)
        
        if hasattr(model_t, 'predict_proba'):
            t_pred = model_t.predict_proba(X_np)[:, 1]  # 获取正类的概率
        else:
            t_pred = model_t.predict(X_np)
    
    # 计算残差
    y_residuals = y_np - y_pred
    t_residuals = t_np - t_pred
    
    # 转换为PyTorch张量
    t_residuals_tensor = torch.tensor(t_residuals, dtype=torch.float32, device=device).reshape(-1, 1)
    y_residuals_tensor = torch.tensor(y_residuals, dtype=torch.float32, device=device)
    
    # 添加截距项
    ones = torch.ones_like(t_residuals_tensor, device=device)
    X_second = torch.cat([ones, t_residuals_tensor], dim=1)
    
    # 使用正规方程求解第二阶段系数
    with torch.no_grad():
        XtX = X_second.t() @ X_second
        Xty = X_second.t() @ y_residuals_tensor.reshape(-1, 1)
        
        # 添加小常数防止奇异矩阵
        XtX += torch.eye(2, device=device) * 1e-6
        
        coefficients = torch.linalg.solve(XtX, Xty)
        effect = coefficients[1].item()  # 处理效应系数
    
    # 计算标准误和置信区间
    if use_robust_se and robust_method == 'bootstrap':
        print("使用自助法计算稳健标准误...")
        se, ci_lower, ci_upper = bootstrap_se_torch(
            X_second.cpu().numpy(), 
            y_residuals_tensor.cpu().numpy(), 
            n_bootstrap=n_bootstrap
        )
        
        # 计算t统计量和p值
        if se > 0:
            t_statistic = effect / se
            n, k = X_second.shape
            df = n - k
            p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))
    else:
        # 计算经典标准误
        n, k = X_second.shape
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            residuals = y_residuals_tensor.reshape(-1, 1) - y_pred_tensor
        mse = torch.sum(residuals**2) / (n - k)
        cov_matrix = torch.linalg.inv(XtX) * mse
        se = torch.sqrt(torch.diag(cov_matrix))[1].item()
        
        # 计算置信区间
        t_stat_val = 1.96  # 95%置信水平的t统计量
        ci_lower = effect - t_stat_val * se
        ci_upper = effect + t_stat_val * se
        
        # 计算t统计量
        t_statistic = effect / se

        # 计算p值（双侧检验）
        df = n - k  # 自由度
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_statistic), df))

        # 计算F统计量
        with torch.no_grad():
            y_pred_tensor = X_second @ coefficients
            ss_residual = torch.sum((y_residuals_tensor.reshape(-1, 1) - y_pred_tensor)**2)
            ss_total = torch.sum((y_residuals_tensor.reshape(-1, 1) - torch.mean(y_residuals_tensor))**2)
            ss_explained = ss_total - ss_residual
    
            # F统计量 = (解释方差/自由度) / (残差方差/自由度)
            f_statistic = (ss_explained / (k - 1)) / (ss_residual / (n - k))
            f_p_value = 1 - stats.f.cdf(f_statistic, k - 1, n - k)

    
    print(f"DML处理效应估计: {effect:.6f}")
    print(f"标准误: {se:.6f}")
    print(f"95%置信区间: [{ci_lower:.6f}, {ci_upper:.6f}]")
    print(f"t统计量: {t_statistic:.4f}")
    print(f"p值: {p_value:.6f}")
    print(f"F统计量: {f_statistic:.4f}")
    print(f"F检验p值: {f_p_value:.6f}")
    
    return {
        'effect': effect,
        'se': se,
        'lb': ci_lower,
        'ub': ci_upper,
        't_statistic': t_statistic,
        'p_value': p_value,
        'f_statistic': f_statistic,
        'f_p_value': f_p_value,
        'df': df
    }

# ========================
# 修改：增强的第一阶段训练函数，尽量使用cuml
# ========================
def first_stage_cuml(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    基础版第一阶段预测函数 - 使用新的评估函数
    """
    clear_gpu_resources()
    
    # 数据分割
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # 创建模型
    model_factory = FirstStageModelFactory()
    
    try:
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        # 特殊处理PyTorch MLP模型
        if model_name == 'MLP':
            # 转换为PyTorch张量
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
            
            # 训练模型
            model.fit(X_train_pt, y_train_pt)
            
            # 预测
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            # 训练其他cuML/XGBoost模型
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        # 使用新的评估函数计算指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
        
        # 根据任务类型打印结果
        if task_type == 'classification':
            print(f"{model_name} {task_type}模型评估 - 准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}")
        else:
            print(f"{model_name} {task_type}模型评估 - R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

def first_stage_cuml_enhanced(X, y, task_type='regression', model_name='RandomForest', test_size=0.25):
    """
    增强版第一阶段预测，支持多种模型 - 使用新的评估函数
    """
    clear_gpu_resources()
    
    # 数据分割 - 使用cuML的train_test_split
    X_train, X_test, y_train, y_test = cu_train_test_split(
        X, y, test_size=test_size, random_state=42
    )
    
    # 创建模型
    model_factory = FirstStageModelFactory()
    
    try:
        # 特殊处理MLP模型，需要输入维度
        if model_name == 'MLP':
            input_size = X_train.shape[1]
            model = model_factory.get_model(model_name, task_type, input_size)
        else:
            model = model_factory.get_model(model_name, task_type)
        
        # 特殊处理PyTorch MLP模型
        if model_name == 'MLP':
            # 转换为PyTorch张量
            X_train_pt = torch.tensor(cp.asnumpy(X_train), dtype=torch.float32)
            y_train_pt = torch.tensor(cp.asnumpy(y_train), dtype=torch.float32)
            
            # 训练模型
            model.fit(X_train_pt, y_train_pt)
            
            # 预测
            X_test_pt = torch.tensor(cp.asnumpy(X_test), dtype=torch.float32)
            y_pred = model.predict(X_test_pt)
            y_pred = cp.array(y_pred)
        else:
            # 训练其他cuML/XGBoost模型
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
        
        # 使用新的评估函数计算评估指标 - 添加task_type参数
        metrics = evaluate_model(model, X_test, y_test, model_name, task_type)
        
        # 添加模型信息
        metrics['model_name'] = model_name
        metrics['task_type'] = task_type
        
        # 根据任务类型输出结果
        if task_type == 'classification':
            summary = f"准确率: {metrics.get('accuracy', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
        else:
            summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
        
        print(f"{model_name} {task_type}模型评估 - {summary}")
        
        return model, metrics
        
    except Exception as e:
        print(f"训练{model_name}模型失败: {str(e)}")
        traceback.print_exc()
        return None, None

# ========================
# 新增：评估指标计算函数（增强版，添加precision, recall）
# ========================
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score
def evaluate_model(model, X_test, y_test, model_name="Model", task_type='classification'):
    """
    评估模型性能，根据任务类型使用不同的指标
    
    参数:
        model: 训练好的模型
        X_test: 测试特征
        y_test: 测试标签
        model_name: 模型名称
        task_type: 任务类型 ('regression' 或 'classification')
    
    返回:
        dict: 包含评估指标的字典
    """
    try:
        # 确保数据是NumPy数组（解决"Implicit conversion to a NumPy array is not allowed"错误）
        if hasattr(X_test, 'get'):
            X_test_np = X_test.get()
        elif hasattr(X_test, 'cpu'):
            X_test_np = X_test.cpu().numpy()
        elif hasattr(X_test, 'numpy'):
            X_test_np = X_test.numpy()
        else:
            X_test_np = np.array(X_test)
            
        if hasattr(y_test, 'get'):
            y_test_np = y_test.get()
        elif hasattr(y_test, 'cpu'):
            y_test_np = y_test.cpu().numpy()
        elif hasattr(y_test, 'numpy'):
            y_test_np = y_test.numpy()
        else:
            y_test_np = np.array(y_test)
        
        # 预测
        y_pred = model.predict(X_test_np)
        
        # 确保预测结果也是NumPy数组
        if hasattr(y_pred, 'get'):
            y_pred_np = y_pred.get()
        elif hasattr(y_pred, 'cpu'):
            y_pred_np = y_pred.cpu().numpy()
        elif hasattr(y_pred, 'numpy'):
            y_pred_np = y_pred.numpy()
        else:
            y_pred_np = np.array(y_pred)
        
        metrics = {}
        
        if task_type == 'regression':
            # 回归任务评估指标
            metrics['r2'] = r2_score(y_test_np, y_pred_np)
            metrics['mse'] = mean_squared_error(y_test_np, y_pred_np)
            metrics['rmse'] = np.sqrt(metrics['mse'])
            metrics['mae'] = mean_absolute_error(y_test_np, y_pred_np)
            
            print(f"{model_name} 回归模型评估 - R²: {metrics['r2']:.4f}, RMSE: {metrics['rmse']:.4f}, MAE: {metrics['mae']:.4f}")
            
        else:
            # 分类任务评估指标
            metrics['accuracy'] = accuracy_score(y_test_np, y_pred_np)
            metrics['precision'] = precision_score(y_test_np, y_pred_np, average='weighted')
            metrics['recall'] = recall_score(y_test_np, y_pred_np, average='weighted')
            metrics['f1'] = f1_score(y_test_np, y_pred_np, average='weighted')
            
            # 计算AUC
            auc_score = np.nan
            try:
                if hasattr(model, 'predict_proba'):
                    y_proba = model.predict_proba(X_test_np)
                    if hasattr(y_proba, 'get'):
                        y_proba = y_proba.get()
                    elif hasattr(y_proba, 'cpu'):
                        y_proba = y_proba.cpu().numpy()
                    
                    if len(y_proba.shape) > 1 and y_proba.shape[1] == 2:  # 二分类
                        auc_score = roc_auc_score(y_test_np, y_proba[:, 1])
                    elif len(y_proba.shape) > 1:  # 多分类
                        auc_score = roc_auc_score(y_test_np, y_proba, multi_class='ovr')
                    else:  # 一维数组
                        auc_score = roc_auc_score(y_test_np, y_proba)
                elif hasattr(model, 'decision_function'):
                    y_score = model.decision_function(X_test_np)
                    if hasattr(y_score, 'get'):
                        y_score = y_score.get()
                    elif hasattr(y_score, 'cpu'):
                        y_score = y_score.cpu().numpy()
                    auc_score = roc_auc_score(y_test_np, y_score)
                else:
                    print(f"警告: 模型 {model_name} 不支持概率预测，无法计算AUC")
                    auc_score = np.nan
            except Exception as e:
                print(f"警告: 计算模型 {model_name} 的AUC时出错: {str(e)}")
                auc_score = np.nan
            
            metrics['auc'] = auc_score
            print(f"{model_name} 分类模型评估 - 准确率: {metrics['accuracy']:.4f}, Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        metrics['predictions'] = y_pred_np
        return metrics
        
    except Exception as e:
        print(f"评估模型 {model_name} 时出错: {str(e)}")
        traceback.print_exc()
        if task_type == 'regression':
            return {
                'r2': np.nan,
                'mse': np.nan,
                'rmse': np.nan,
                'mae': np.nan,
                'predictions': None
            }
        else:
            return {
                'accuracy': np.nan,
                'precision': np.nan,
                'recall': np.nan,
                'f1': np.nan,
                'auc': np.nan,
                'predictions': None
            }

# ========================
# 新增：模型比较器类
# ========================
class ModelComparator:
    """模型比较器，用于比较不同模型的性能并选择最佳模型"""
    
    def __init__(self, task_type):
        self.task_type = task_type
        self.model_metrics = {}
        self.weights = self._get_default_weights()
    
    def add_model(self, model_name, metrics):
        """添加模型及其评估指标"""
        self.model_metrics[model_name] = metrics
    
    def _get_default_weights(self):
        """获取默认指标权重"""
        if self.task_type == 'classification':
            return {'accuracy': 0.2, 'precision': 0.2, 'recall': 0.2, 'f1': 0.2, 'auc': 0.2}
        else:
            return {'r2': 0.4, 'rmse': 0.3, 'mae': 0.3}
    
    def _calculate_score(self, metrics):
        """计算模型综合得分（修复版）"""
        if self.task_type == 'regression':
            # 以R²为主要依据，同时考虑RMSE和MAE（数值越小越好，故取倒数）
            r2 = metrics.get('r2', 0)
            rmse = metrics.get('rmse', 1e6)
            mae = metrics.get('mae', 1e6)
            # 如果R²为负，给予惩罚性低分
            if r2 < 0:
                return -1.0 * abs(r2)
            # 综合得分，R²权重最高
            score = r2 * 0.7 + (1 / (rmse + 1e-6)) * 0.15 + (1 / (mae + 1e-6)) * 0.15
            return score
        else:
            # 分类任务：可以考虑准确率、F1、AUC的加权平均
            accuracy = metrics.get('accuracy', 0)
            precision = metrics.get('precision', 0)
            recall = metrics.get('recall', 0)
            f1 = metrics.get('f1', 0)
            auc = metrics.get('auc', 0)
            if np.isnan(auc):
                auc = 0
            # 赋予AUC较高权重，如果AUC为0则主要依赖准确率和F1
            if auc > 0:
                score = accuracy * 0.2 + precision * 0.2 + recall * 0.2 + f1 * 0.2 + auc * 0.2
            else:
                score = accuracy * 0.5 + f1 * 0.5
            return score
    
    def compare_models(self):
        """比较所有模型并返回排名"""
        model_scores = {}
        
        for model_name, metrics in self.model_metrics.items():
            score = self._calculate_score(metrics)
            model_scores[model_name] = score
        
        # 按得分排序
        sorted_models = sorted(model_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_models
    
    def get_best_model(self):
        """获取最佳模型名称"""
        sorted_models = self.compare_models()
        return sorted_models[0][0] if sorted_models else None
    
    def print_comparison(self):
        """打印模型比较结果 - 使用新的评估函数"""
        print(f"\n=== {self.task_type.upper()} 模型比较结果 ===")
        sorted_models = self.compare_models()
        
        for rank, (model_name, score) in enumerate(sorted_models, 1):
            metrics = self.model_metrics[model_name]
            
            # 使用新的评估函数格式输出
            if self.task_type == 'classification':
                summary = f"准确率: {metrics.get('accuracy', 0):.4f}, Precision: {metrics.get('precision', 0):.4f}, Recall: {metrics.get('recall', 0):.4f}, F1: {metrics.get('f1', 0):.4f}, AUC: {metrics.get('auc', 0):.4f}"
            else:
                summary = f"R²: {metrics.get('r2', 0):.4f}, RMSE: {metrics.get('rmse', 0):.4f}, MAE: {metrics.get('mae', 0):.4f}"
            
            print(f"{rank}. {model_name}: 得分={score:.4f}, {summary}")

# ========================
# 敏感性分析类（优化版）
# ========================
class SensitivityAnalysisOptimized:
    """敏感性分析类（优化版）"""
    
    def __init__(self, X: np.ndarray, y: np.ndarray, t: np.ndarray, control_variables: list):
        self.X = X
        self.y = y
        self.t = t
        self.control_variables = control_variables
        self.results = {}

    def run_analysis(self, variations: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
        """运行敏感性分析 - 确保处理所有场景"""
        for name, variation in variations.items():
            print(f"运行敏感性分析: {name}")
            
            try:
                # 应用变异
                X_var = self.X
                if 'control_vars' in variation:
                    control_indices = [i for i, var in enumerate(self.control_variables) 
                                     if var in variation['control_vars']]
                    X_var = self.X[:, control_indices]
                
                print("训练第一阶段模型...")
                
                # 使用增强版函数训练第一阶段模型
                model_y, metrics_y = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.y), 
                    task_type='regression', 
                    model_name=variation.get('model_name', 'RandomForest')
                )
                model_t, metrics_t = first_stage_cuml_enhanced(
                    cp.array(X_var), cp.array(self.t), 
                    task_type='classification', 
                    model_name=variation.get('model_name', 'RandomForest')
                )
                
                # 计算残差（DML核心步骤）
                y_pred = model_y.predict(X_var)
                t_pred = model_t.predict(X_var)
                
                y_residuals = self.y - y_pred
                t_residuals = self.t - t_pred
                
                # 使用正确的第二阶段DML分析函数
                dml_result = second_stage_dml_analysis(
                    t_residuals=t_residuals,
                    y_residuals=y_residuals,
                    robust_method=variation.get('robust_method', 'bootstrap'),
                    n_bootstrap=variation.get('n_bootstrap', 1000)
                )
                
                # 提取所有结果，包括统计检验指标
                effect = dml_result['effect']
                se = dml_result['se']
                lb = dml_result['lb']
                ub = dml_result['ub']
                t_statistic = dml_result.get('t_statistic', np.nan)
                p_value = dml_result.get('p_value', np.nan)
                f_statistic = dml_result.get('f_statistic', np.nan)
                f_p_value = dml_result.get('f_p_value', np.nan)
                
                self.results[name] = {
                    'effect': effect,
                    'se': se,
                    'lb': lb,
                    'ub': ub,
                    't_statistic': t_statistic,
                    'p_value': p_value,
                    'f_statistic': f_statistic,
                    'f_p_value': f_p_value,
                    'metrics_y': metrics_y,
                    'metrics_t': metrics_t,
                    'variation': variation
                }
                
                print(f"{name}: 效应={effect:.6f}, 标准误={se:.6f}")
                
            except Exception as e:
                print(f"敏感性分析 {name} 出错: {str(e)}")
                traceback.print_exc()
                self.results[name] = {'error': str(e)}
        
        return self.results
    
    def summarize(self):
        """
        总结并格式化输出敏感性分析结果
        """
        summary_lines = []
        summary_lines.append("\n=== 敏感性分析结果总结 ===")
        summary_lines.append("场景\t\t处理效应\t标准误\t95%置信区间")
        summary_lines.append("-" * 65)
        
        for name, result in self.results.items():
            if 'error' not in result:
                effect = result['effect']
                se = result['se']
                lb = result['lb']
                ub = result['ub']
                summary_lines.append(f"{name[:12]:<12}\t{effect:.6f}\t{se:.6f}\t[{lb:.6f}, {ub:.6f}]")
            else:
                summary_lines.append(f"{name[:12]:<12}\t分析出错: {result['error']}")
        
        # 计算效应变化范围（如果所有分析都成功）
        successful_results = [res for res in self.results.values() if 'error' not in res]
        if len(successful_results) > 0:
            effects = [res['effect'] for res in successful_results]
            min_effect = min(effects)
            max_effect = max(effects)
            range_effect = max_effect - min_effect
            summary_lines.append("-" * 65)
            summary_lines.append(f"处理效应范围: {min_effect:.6f} 到 {max_effect:.6f} (范围: {range_effect:.6f})")
            summary_lines.append(f"效应变化幅度: {(range_effect / np.mean(effects)) * 100:.2f}%")
        
        return "\n".join(summary_lines)

# ========================
# 数据预处理封装
# ========================
class DataPreprocessor:
    """数据预处理类，封装数据读取和预处理逻辑"""
    
    def __init__(self, control_variables):
        self.control_variables = control_variables
        self.scalers = {}
        self.imputers = {}
        self.encoders = {}
    
    def load_and_preprocess_data(self, file_path, y_col, t_col):
        """加载并预处理数据"""
        print("步骤1: 数据准备")
        print(f"使用文件: {file_path}")
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"文件 '{file_path}' 不存在")
        
        # 读取数据
        data = pd.read_stata(file_path)
        
        # 处理因变量和处理变量
        y_series = data[y_col]
        t_series = data[t_col]
        
        # 编码和预处理
        le = LabelEncoder()
        y = le.fit_transform(y_series) if y_series.dtype == 'object' or isinstance(y_series.dtype, CategoricalDtype) else y_series.values
        t = le.fit_transform(t_series) if t_series.dtype == 'object' or isinstance(t_series.dtype, CategoricalDtype) else t_series.values

        # 检查所有控制变量是否存在于数据中
        missing_cols = [col for col in self.control_variables if col not in data.columns]
        if missing_cols:
            raise ValueError(f"数据中缺少以下控制变量: {missing_cols}")
        
        # 预处理控制变量
        for col in self.control_variables:
            # 处理分类变量
            if data[col].dtype == 'object' or isinstance(data[col].dtype, CategoricalDtype):
                self.encoders[col] = LabelEncoder()
                data[col] = self.encoders[col].fit_transform(data[col].astype(str))
            
            # 处理缺失值
            if data[col].isnull().sum() > 0:
                self.imputers[col] = SimpleImputer(strategy='mean')
                data[col] = self.imputers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()
            
            # 特征缩放
            self.scalers[col] = StandardScaler()
            data[col] = self.scalers[col].fit_transform(data[col].values.reshape(-1, 1)).flatten()

        # 创建特征矩阵
        X = data[self.control_variables].values
        
        # 转换为GPU数组
        X_gpu = cp.array(X, dtype=cp.float32)
        y_gpu = cp.array(y, dtype=cp.float32)
        t_gpu = cp.array(t, dtype=cp.float32)
        
        # 检查数据完整性
        print("检查数据完整性：")
        print("X 中是否有 NaN 或 Inf:", cp.any(cp.isnan(X_gpu)), cp.any(cp.isinf(X_gpu)))
        print("y 中是否有 NaN 或 Inf:", cp.any(cp.isnan(y_gpu)), cp.any(cp.isinf(y_gpu)))
        print("t 中是否有 NaN 或 Inf:", cp.any(cp.isnan(t_gpu)), cp.any(cp.isinf(t_gpu)))
        
        # 处理缺失值或无穷值
        X_gpu = cp.nan_to_num(X_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        y_gpu = cp.nan_to_num(y_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        t_gpu = cp.nan_to_num(t_gpu, nan=0.0, posinf=0.0, neginf=0.0)
        
        print(f"数据形状: X={X_gpu.shape}, y={y_gpu.shape}, t={t_gpu.shape}")
        
        return cp.asnumpy(X_gpu), cp.asnumpy(y_gpu), cp.asnumpy(t_gpu), data

# ========================
# 新增：使用5折交叉验证计算残差
# ========================
def compute_residuals_with_cv(model, X, y, task_type='regression', n_folds=5):
    """
    使用5折交叉验证计算残差
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    residuals = np.zeros(len(y))
    predictions = np.zeros(len(y))
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # 克隆模型以避免状态污染
        cloned_model = clone(model)
        cloned_model.fit(X_train, y_train)
        
        if task_type == 'classification' and hasattr(cloned_model, 'predict_proba'):
            y_pred_val = cloned_model.predict_proba(X_val)[:, 1]
        else:
            y_pred_val = cloned_model.predict(X_val)
        
        residuals[val_idx] = y_val - y_pred_val
        predictions[val_idx] = y_pred_val
    
    return residuals, predictions

# ========================
# 结果保存函数（增强版）
# ========================
def save_detailed_results(effect, se, lb, ub, t_statistic, p_value, f_statistic, f_p_value, model_y, model_t, metrics_y, metrics_t,
                         sensitivity_results, save_path, robust_method='bootstrap',
                         batch_id="batch_001"):
    """
    保存详细的结果到文件（修复版）- 确保敏感性分析结果正确合并
    """
    # 生成时间戳
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 保存主要结果到CSV
    results_df = pd.DataFrame({
        'timestamp': [datetime.now()],
        'batch_id': [batch_id],
        'effect': [effect],
        'se': [se],
        'lb': [lb],
        'ub': [ub],
        't_statistic': [t_statistic],
        'p_value': [p_value],
        'f_statistic': [f_statistic],
        'f_p_value': [f_p_value],
        'model_y_type': [type(model_y).__name__],
        'model_t_type': [type(model_t).__name__],
        # Y模型指标
        'y_r2': [metrics_y.get('r2', np.nan)],
        'y_mse': [metrics_y.get('mse', np.nan)],
        'y_rmse': [metrics_y.get('rmse', np.nan)],
        'y_mae': [metrics_y.get('mae', np.nan)],
        # T模型指标
        't_accuracy': [metrics_t.get('accuracy', np.nan)],
        't_precision': [metrics_t.get('precision', np.nan)],
        't_recall': [metrics_t.get('recall', np.nan)],
        't_f1': [metrics_t.get('f1', np.nan)],
        't_auc': [metrics_t.get('auc', np.nan)],
        'robust_method': [robust_method],
        'analysis_type': ['main'],
        'variation': ['base_model']
    })
    
    # 保存敏感性分析结果 - 添加更健壮的处理逻辑
    sensitivity_data = []
    for name, result in sensitivity_results.items():
        if 'error' not in result:
            sens_row = {
                'timestamp': datetime.now(),
                'batch_id': batch_id,
                'effect': result.get('effect', np.nan),
                'se': result.get('se', np.nan),
                'lb': result.get('lb', np.nan),
                'ub': result.get('ub', np.nan),
                't_statistic': result.get('t_statistic', np.nan),
                'p_value': result.get('p_value', np.nan),
                'f_statistic': result.get('f_statistic', np.nan),
                'f_p_value': result.get('f_p_value', np.nan),
                'y_r2': result.get('metrics_y', {}).get('r2', np.nan),
                'y_mse': result.get('metrics_y', {}).get('mse', np.nan),
                'y_rmse': result.get('metrics_y', {}).get('rmse', np.nan),
                'y_mae': result.get('metrics_y', {}).get('mae', np.nan),
                't_accuracy': result.get('metrics_t', {}).get('accuracy', np.nan),
                't_precision': result.get('metrics_t', {}).get('precision', np.nan),
                't_recall': result.get('metrics_t', {}).get('recall', np.nan),
                't_f1': result.get('metrics_t', {}).get('f1', np.nan),
                't_auc': result.get('metrics_t', {}).get('auc', np.nan),
                'robust_method': result.get('variation', {}).get('robust_method', robust_method),
                'analysis_type': 'sensitivity',
                'variation': name
            }
            sensitivity_data.append(sens_row)
        else:
            print(f"警告: 敏感性分析场景 '{name}' 包含错误: {result['error']}")
    
    # 创建敏感性分析DataFrame
    if sensitivity_data:
        sensitivity_df = pd.DataFrame(sensitivity_data)
    else:
        print("警告: 没有可用的敏感性分析数据")
        sensitivity_df = pd.DataFrame()
    
    # 合并主要结果和敏感性分析结果
    if not sensitivity_df.empty:
        combined_results = pd.concat([results_df, sensitivity_df], ignore_index=True)
    else:
        combined_results = results_df
        print("警告: 仅保存主要分析结果，无敏感性分析数据")
    
    # 确保保存目录存在
    os.makedirs(save_path, exist_ok=True)
    
    # 保存合并后的结果
    combined_file = os.path.join(save_path, f"combined_analysis_results_{timestamp_str}.csv")
    combined_results.to_csv(combined_file, index=False)
    
    # 同时单独保存敏感性分析结果（如果有数据）
    if not sensitivity_df.empty:
        sensitivity_file = os.path.join(save_path, f"sensitivity_analysis_{timestamp_str}.csv")
        sensitivity_df.to_csv(sensitivity_file, index=False)
        print(f"详细结果已保存到:")
        print(f"- 合并结果: {combined_file}")
        print(f"- 敏感性分析: {sensitivity_file}")
    else:
        print(f"主要结果已保存到: {combined_file}")
        print("注意: 无敏感性分析数据可保存")
    
    return combined_file

# ========================
# 新增：绘制指标散点图函数（修改版）
# ========================
def plot_metrics_scatter(y_metrics_dict, t_metrics_dict, group_name, save_path):
    """
    绘制Y模型和T模型的指标散点图
    """
    # 提取模型名称（假设Y和T模型相同）
    models = list(y_metrics_dict.keys())
    
    # Y模型指标（回归）
    y_metrics_names = ['R²', 'RMSE', 'MAE']
    y_metrics_df = pd.DataFrame({
        'Model': models,
        'R²': [y_metrics_dict[m].get('r2', np.nan) for m in models],
        'RMSE': [y_metrics_dict[m].get('rmse', np.nan) for m in models],
        'MAE': [y_metrics_dict[m].get('mae', np.nan) for m in models]
    })
    
    # T模型指标（分类）
    t_metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
    t_metrics_df = pd.DataFrame({
        'Model': models,
        'Accuracy': [t_metrics_dict[m].get('accuracy', np.nan) for m in models],
        'Precision': [t_metrics_dict[m].get('precision', np.nan) for m in models],
        'Recall': [t_metrics_dict[m].get('recall', np.nan) for m in models],
        'F1': [t_metrics_dict[m].get('f1', np.nan) for m in models],
        'AUC': [t_metrics_dict[m].get('auc', np.nan) for m in models]
    })
    
    # 创建一个图形和子图，增加整体高度以容纳更多间距
    fig = plt.figure(figsize=(8, 6))  # 缩小整体尺寸

    # 使用 gridspec 设置子图布局，按指标数量比例分配高度，并设置无间隔
    gs = gridspec.GridSpec(2, 1, height_ratios=[3, 5], hspace=0.0)  # hspace=0.0 去除间隔

    # 创建两个子图
    ax1 = fig.add_subplot(gs[0])  # 第一个子图 (Y model)
    ax2 = fig.add_subplot(gs[1], sharex=ax1)  # 第二个子图 (T model)，共享X轴

    # 关闭上子图的X轴标签
    ax1.tick_params(labelbottom=False)

    # 设置X轴标签（共享，只在底部显示）
    ax2.set_xlabel('Model', fontsize=12, fontweight='bold')

    # 明确设置X轴刻度和标签，确保所有模型都显示在横坐标上
    x_positions = range(len(models))
    ax2.set_xticks(x_positions)
    ax2.set_xticklabels(models, rotation=0, ha='center', fontweight='bold', color='black')  # 水平、黑色加粗

    # 设置Y轴范围（保持0-1）
    for ax in [ax1, ax2]:
        ax.set_ylim(0, 1)

    # Y模型位置（3个指标，再次缩小间距：从0.8到0.2）
    y_positions = np.linspace(0.8, 0.2, len(y_metrics_names))  # 再次缩小范围
    
    # T模型位置（5个指标，再次缩小间距范围）
    t_positions = np.linspace(0.8, 0.2, len(t_metrics_names))  # 修改：再次缩小范围到0.8-0.2
    
    # 颜色：使用viridis colormap
    y_colors = cm.viridis(np.linspace(0, 1, len(y_metrics_names)))
    t_colors = cm.viridis(np.linspace(0, 1, len(t_metrics_names)))
    
    # 绘制横向参考线（Y轴参考线）
    for ax, positions in zip([ax1, ax2], [y_positions, t_positions]):
        for y in positions:
            ax.axhline(y=y, color='white', linestyle='--', linewidth=2, zorder=1)
    
    # 绘制纵向参考线（X轴参考线）
    for ax in [ax1, ax2]:
        for i in x_positions:
            ax.axvline(x=i, color='white', linestyle='--', linewidth=2, zorder=1)
    
    # 设置背景色
    ax1.set_facecolor('#F0F0F0')  # 淡灰色
    ax2.set_facecolor('#E6E6FA')  # 淡紫色
    
    max_size = 800  
    for i, metric in enumerate(y_metrics_names):
        values = y_metrics_df[metric].values
        if np.all(np.isnan(values)):
            continue  # 跳过全NaN
        valid_mask = ~np.isnan(values)
        valid_values = values[valid_mask]
        if len(valid_values) == 0:
            continue
        
        if metric == 'R²': 
            norm_values = (valid_values - valid_values.min()) / (valid_values.max() - valid_values.min() + 1e-6)
            sizes = (norm_values ** 5) * max_size + 50  
        else: 
            norm_values = (valid_values.max() - valid_values) / (valid_values.max() - valid_values.min() + 1e-6)
            sizes = (norm_values ** 5) * max_size + 50  
        
        # 绘制（只绘制有效值）
        ax1.scatter(np.array(x_positions)[valid_mask], [y_positions[i]] * len(valid_values), s=sizes, color=y_colors[i], alpha=0.7, label=metric)
    
    # 为T模型绘制散点（高度敏感映射，所有正向）
    for i, metric in enumerate(t_metrics_names):
        values = t_metrics_df[metric].values
        if np.all(np.isnan(values)):
            continue
        valid_mask = ~np.isnan(values)
        valid_values = values[valid_mask]
        if len(valid_values) == 0:
            continue
        
        norm_values = (valid_values - valid_values.min()) / (valid_values.max() - valid_values.min() + 1e-6)
        sizes = (norm_values ** 5) * max_size + 50  # 修改：5次方放大差异，提高敏感度
        ax2.scatter(np.array(x_positions)[valid_mask], [t_positions[i]] * len(valid_values), s=sizes, color=t_colors[i], alpha=0.7, label=metric)
    
    # 设置Y轴标签
    ax1.set_yticks(y_positions)
    ax1.set_yticklabels(y_metrics_names, fontweight='bold', color='black')
    ax2.set_yticks(t_positions)
    ax2.set_yticklabels(t_metrics_names, fontweight='bold', color='black')
    
    # 设置Y标签
    ax1.set_ylabel('Y model', fontsize=15, fontweight='bold', color='blue')
    ax2.set_ylabel('T model', fontsize=15, fontweight='bold', color='purple')
    
    # 关闭图例（可选，如果需要可开启）
    ax1.legend().set_visible(False)
    ax2.legend().set_visible(False)
    
    plt.tight_layout()
    fig_path = os.path.join(save_path, f"{group_name}_metrics_plot.pdf")
    plt.savefig(fig_path, format='pdf', bbox_inches='tight', dpi=1200)
    plt.close()
    print(f"指标散点图已保存到: {fig_path}")

# ========================
# 新增：保存到Word文档函数
# ========================
def save_to_word(group_name, dml_results, best_y_model_name, best_metrics_y, best_t_model_name, best_metrics_t, sensitivity_summary, save_path):
    """
    将结果保存到Word文档
    """
    doc = Document()
    doc.add_heading(f'{group_name} 分析结果', level=1)
    
    # 添加主要结果
    p = doc.add_paragraph()
    p.add_run('最终结果:\n').bold = True
    p.add_run(f"处理效应估计: {dml_results['effect']}\n")
    p.add_run(f"标准误差: {dml_results['se']}\n")
    p.add_run(f"置信区间下界: {dml_results['lb']}\n")
    p.add_run(f"置信区间上界: {dml_results['ub']}\n")
    p.add_run(f"p值: {dml_results['p_value']}\n")
    p.add_run(f"t检验值: {dml_results['t_statistic']}\n")
    p.add_run(f"F检验值: {dml_results['f_statistic']}\n")
    p.add_run(f"f检验P值: {dml_results['f_p_value']}\n")
    
    # 添加最佳模型
    p = doc.add_paragraph()
    p.add_run('最佳Y模型: ').bold = True
    p.add_run(f"{best_y_model_name}\n")
    p.add_run(f"R²: {best_metrics_y.get('r2', np.nan)}\n")
    p.add_run(f"RMSE: {best_metrics_y.get('rmse', np.nan)}\n")
    p.add_run(f"MAE: {best_metrics_y.get('mae', np.nan)}\n")
    
    p = doc.add_paragraph()
    p.add_run('最佳T模型: ').bold = True
    p.add_run(f"{best_t_model_name}\n")
    p.add_run(f"Accuracy: {best_metrics_t.get('accuracy', np.nan)}\n")
    p.add_run(f"Precision: {best_metrics_t.get('precision', np.nan)}\n")
    p.add_run(f"Recall: {best_metrics_t.get('recall', np.nan)}\n")
    p.add_run(f"F1: {best_metrics_t.get('f1', np.nan)}\n")
    p.add_run(f"AUC: {best_metrics_t.get('auc', np.nan)}\n")
    
    # 添加敏感性分析总结
    p = doc.add_paragraph()
    p.add_run('敏感性分析总结:\n').bold = True
    p.add_run(sensitivity_summary)
    
    # 保存文档
    word_path = os.path.join(save_path, f"{group_name}_results.docx")
    doc.save(word_path)
    print(f"结果已保存到Word: {word_path}")

# ========================
# 修改：主程序逻辑（增强版，处理多个组）
# ========================
if __name__ == "__main__":
    try:
        # 初始化时限制GPU内存
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.8)
            
        print("正在执行DML处理流程（多模型比较版）")
        print(f"cuML版本: {cuml_version}")
        
        # 定义组配置
        groups = [
            {
                'name': '反学习俘获组',
                'y_col': 'anti_stucap',
                't_col': 'mobile_use',
                'control_groups': {
                    1: [
                        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                        'intelligence', 'person_status', 'party', 'workasso',
                        'fid','pid','year',
                        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                        'intelligence_sq', 'person_status_sq', 
                        'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                        'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
                        'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
                        'intelligence_cub', 'person_status_cub'
                    ]
                }
            },
            {
                'name': '反社交俘获组',
                'y_col': 'anti_socicap',
                't_col': 'mobile_use',
                'control_groups': {
                    1: [
                        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                        'intelligence', 'person_status', 'party', 'workasso',
                        'fid','pid','year',
                        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                        'intelligence_sq', 'person_status_sq', 
                        'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                        'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
                        'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
                        'intelligence_cub', 'person_status_cub'
                    ]
                }
            },
            {
                'name': '反娱乐俘获组',
                'y_col': 'anti_playcap',
                't_col': 'mobile_use',
                'control_groups': {
                    1: [
                        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                        'intelligence', 'person_status', 'party', 'workasso',
                        'fid','pid','year',
                        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                        'intelligence_sq', 'person_status_sq', 
                        'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                        'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
                        'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
                        'intelligence_cub', 'person_status_cub'
                    ]
                }
            },
            {
                'name': '反资金俘获组',
                'y_col': 'anti_fundscap',
                't_col': 'mobile_use',
                'control_groups': {
                    1: [
                        'contracts', 'ln_total_asset', 'impro_sasti', 'Child',
                        'lifefire', 'age', 'ln_renqing', 'cfpsedu', 'laborasso', 'familysize', 
                        'Oldage', 'health', 'marriage', 'army', 'ln_finc', 'exercise', 
                        'intelligence', 'person_status', 'party', 'workasso',
                        'fid','pid','year',
                        'ln_total_asset_sq', 'impro_sasti_sq', 'Child_sq',
                        'lifefire_sq', 'age_sq', 'ln_renqing_sq', 'cfpsedu_sq',  'familysize_sq', 
                        'Oldage_sq', 'health_sq', 'marriage_sq', 'ln_finc_sq', 'exercise_sq', 
                        'intelligence_sq', 'person_status_sq', 
                        'ln_total_asset_cub', 'impro_sasti_cub', 'Child_cub',
                        'lifefire_cub', 'age_cub', 'ln_renqing_cub', 'cfpsedu_cub',  'familysize_cub', 
                        'Oldage_cub', 'health_cub', 'marriage_cub', 'ln_finc_cub', 'exercise_cub', 
                        'intelligence_cub', 'person_status_cub'
                    ]
                }
            }
        ]
        
        # 保存路径
        save_path = "D:\\my-rapids-project\\result"
        os.makedirs(save_path, exist_ok=True)
        
        # 定义要比较的模型列表
        model_names = ['RandomForest', 'LogisticRegression', 'SVC', 'XGBoost', 'MLP']
        if HAS_GRADIENT_BOOSTING:
            model_names.append('GradientBoosting')
        
        # 循环处理每个组
        for group in groups:
            group_name = group['name']
            y_col = group['y_col']
            t_col = group['t_col']
            control_names = group['control_groups'][1]  # 使用组1变量
            
            print(f"\n=== 处理组: {group_name} ===")
            
            # 数据预处理
            preprocessor = DataPreprocessor(control_names)
            X, y, t, original_data = preprocessor.load_and_preprocess_data("l3dml_data_digital.dta", y_col, t_col)
            
            # 转换为GPU数组
            X_gpu = cp.array(X, dtype=cp.float32)
            y_gpu = cp.array(y, dtype=cp.float32)
            t_gpu = cp.array(t, dtype=cp.float32)
            
            # 创建模型比较器
            y_comparator = ModelComparator('regression')
            t_comparator = ModelComparator('classification')
            
            # 训练并比较Y模型（回归任务）
            print("\n=== 训练并比较Y模型（回归任务）===")
            y_models = {}
            y_metrics_dict = {}
            for model_name in model_names:
                try:
                    print(f"\n训练Y模型: {model_name}")
                    model, metrics = first_stage_cuml_enhanced(
                        X_gpu, y_gpu, 
                        task_type='regression',
                        model_name=model_name
                    )
                    if model is not None and metrics is not None:
                        y_models[model_name] = model
                        y_metrics_dict[model_name] = metrics
                        y_comparator.add_model(model_name, metrics)
                except Exception as e:
                    print(f"训练Y模型 {model_name} 失败: {str(e)}")
                    traceback.print_exc()
            
            # 训练并比较T模型（分类任务）
            print("\n=== 训练并比较T模型（分类任务）===")
            t_models = {}
            t_metrics_dict = {}
            for model_name in model_names:
                try:
                    print(f"\n训练T模型: {model_name}")
                    model, metrics = first_stage_cuml_enhanced(
                        X_gpu, t_gpu, 
                        task_type='classification', 
                        model_name=model_name
                    )
                    if model is not None and metrics is not None:
                        t_models[model_name] = model
                        t_metrics_dict[model_name] = metrics
                        t_comparator.add_model(model_name, metrics)
                except Exception as e:
                    print(f"训练T模型 {model_name} 失败: {str(e)}")
                    traceback.print_exc()
            
            # 打印比较结果
            y_comparator.print_comparison()
            t_comparator.print_comparison()
            
            # 绘制指标散点图
            print(f"\n绘制 {group_name} 指标散点图...")
            plot_metrics_scatter(y_metrics_dict, t_metrics_dict, group_name, save_path)
            
            # 选择最佳模型
            best_y_model_name = y_comparator.get_best_model()
            best_t_model_name = t_comparator.get_best_model()
            
            if best_y_model_name is None or best_t_model_name is None:
                raise ValueError("未能成功训练任何模型，无法进行DML分析")
            
            print(f"\n选择的最佳Y模型: {best_y_model_name}")
            print(f"选择的最佳T模型: {best_t_model_name}")
            
            # 获取最佳模型
            best_model_y = y_models[best_y_model_name]
            best_model_t = t_models[best_t_model_name]
            best_metrics_y = y_comparator.model_metrics[best_y_model_name]
            best_metrics_t = t_comparator.model_metrics[best_t_model_name]
            
            # 如果最佳模型不存在，重新训练
            if best_model_y is None:
                print(f"重新训练Y模型: {best_y_model_name}")
                best_model_y, _ = first_stage_cuml_enhanced(
                    X_gpu, y_gpu, 
                    task_type='regression',
                    model_name=best_y_model_name
                )
            
            if best_model_t is None:
                print(f"重新训练T模型: {best_t_model_name}")
                best_model_t, _ = first_stage_cuml_enhanced(
                    X_gpu, t_gpu, 
                    task_type='classification', 
                    model_name=best_t_model_name
                )
            
            # 使用5折CV计算残差
            print("\n使用5折交叉验证计算残差...")
            y_residuals, y_pred = compute_residuals_with_cv(best_model_y, X, y, task_type='regression')
            t_residuals, t_pred = compute_residuals_with_cv(best_model_t, X, t, task_type='classification')
            
            # 执行完整的第二阶段DML分析
            print("\n步骤3: 执行完整的第二阶段DML分析")
            dml_results = second_stage_dml_analysis(
                t_residuals=t_residuals,
                y_residuals=y_residuals,
                robust_method='bootstrap',
                n_bootstrap=1000
            )
            
            # 执行敏感性分析（使用优化版）
            print("\n步骤4: 执行敏感性分析（优化版）")
            sensitivity_analyzer = SensitivityAnalysisOptimized(
                X, y, t, control_names
            )
            
            # 定义敏感性分析场景
            variations = {
                "基准模型": {
                    "control_vars": control_names,
                    "use_robust_se": True,
                    "robust_method": "bootstrap"
                },
                "简化控制变量": {
                    "control_vars": ['contracts', 'ln_total_asset', 'age', 'cfpsedu', 'familysize'],
                    "use_robust_se": True,
                    "robust_method": "bootstrap"
                },
                "不同稳健方法": {
                    "control_vars": control_names,
                    "use_robust_se": True,
                    "robust_method": "hc1"
                }
            }
            
            # 运行敏感性分析
            sensitivity_results = sensitivity_analyzer.run_analysis(variations)
            
            # 总结敏感性分析结果
            try:
                sensitivity_summary = sensitivity_analyzer.summarize()
                print(sensitivity_summary)
            except AttributeError as e:
                print(f"警告: 无法生成敏感性分析总结，方法可能未实现: {e}")
                sensitivity_summary = "总结不可用"
            except Exception as e:
                print(f"生成敏感性分析总结时出错: {e}")
                sensitivity_summary = "总结生成出错"
            
            # 输出结果
            print("\n最终结果:")
            print("处理效应估计:", dml_results['effect'])
            print("标准误差:", dml_results['se'])
            print("置信区间下界:", dml_results['lb'])
            print("置信区间上界:", dml_results['ub'])
            print("p值:", dml_results['p_value'])
            print("t检验值:", dml_results['t_statistic'])
            print("F检验值:", dml_results['f_statistic'])
            print("f检验P值:", dml_results['f_p_value'])
            
            # 保存结果到CSV
            print("\n步骤5: 保存结果到CSV")
            save_detailed_results(
                dml_results['effect'], dml_results['se'], dml_results['lb'], dml_results['ub'], 
                dml_results['t_statistic'], dml_results['p_value'], dml_results['f_statistic'], dml_results['f_p_value'],
                best_model_y, best_model_t, 
                best_metrics_y, best_metrics_t,
                sensitivity_results, 
                save_path, 
                robust_method='bootstrap',
                batch_id=group_name
            )
            
            # 保存到Word
            print("保存结果到Word...")
            save_to_word(group_name, dml_results, best_y_model_name, best_metrics_y, best_t_model_name, best_metrics_t, sensitivity_summary, save_path)
        
        print("所有组分析完成！结果已保存。")
        
    except Exception as e:
        print(f"程序执行出错: {str(e)}")
        traceback.print_exc()
        with open(os.path.join(save_path, "error.log"), "a", encoding='utf-8') as log_file:
            log_file.write(f"[{datetime.now()}] 错误信息: {str(e)}\n")
        
    finally:
        print("执行终极资源清理...")
        clear_gpu_resources()
        
        if torch.cuda.is_available():
            print("最终GPU内存状态:")
            print(f"已用内存: {torch.cuda.memory_allocated()/1e9:.2f} GB")
            print(f"保留内存: {torch.cuda.memory_reserved()/1e9:.2f} GB")
            
        print("资源清理完成，程序退出！")


/opt/conda/envs/rapids-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuML Gradient Boosting 不可用，将跳过该模型
正在执行DML处理流程（多模型比较版）
cuML版本: 25.08.00

=== 处理组: 反学习俘获组 ===
步骤1: 数据准备
使用文件: l3dml_data_digital.dta
检查数据完整性：
X 中是否有 NaN 或 Inf: False False
y 中是否有 NaN 或 Inf: False False
t 中是否有 NaN 或 Inf: False False
数据形状: X=(26354, 53), y=(26354,), t=(26354,)

=== 训练并比较Y模型（回归任务）===

训练Y模型: RandomForest
RandomForest 回归模型评估 - R²: 0.4299, RMSE: 1.2444, MAE: 0.9103
RandomForest regression模型评估 - R²: 0.4299, RMSE: 1.2444, MAE: 0.9103

训练Y模型: LogisticRegression
LogisticRegression 回归模型评估 - R²: 0.4132, RMSE: 1.2624, MAE: 1.0117
LogisticRegression regression模型评估 - R²: 0.4132, RMSE: 1.2624, MAE: 1.0117

训练Y模型: SVC
SVC 回归模型评估 - R²: 0.3893, RMSE: 1.2879, MAE: 0.8705
SVC regression模型评估 - R²: 0.3893, RMSE: 1.2879, MAE: 0.8705

训练Y模型: XGBoost
XGBoost 回归模型评估 - R²: 0.4009, RMSE: 1.2757, MAE: 0.9400
XGBoost regression模型评估 - R²: 0.4009, RMSE: 1.2757, MAE: 0.9400

训练Y模型: MLP
Epoch [20/100], Loss: 2.7322
Epoch [40/100], Loss: 2.7264
Epoch [60/100], Loss: 2.7221
Epoch [80/100], Loss: 2.7212
Epo